<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_05_02_00_meta_learners_introduction_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1dB5MWviA5QpiG_oSmt6ZLejPA6waRFnM)

# 5.2 Meta-Learners for Treatment Effect Estimation

In many real-world settings, a treatment (like a drug, policy, or recommendation) doesn't affect everyone the same way. **Meta-learners** are a family of algorithms designed to estimate *how much* a treatment affects different individuals — a quantity known as the **Conditional Average Treatment Effect (CATE)**:

$$\tau(x) = \mathbb{E}[Y(1) - Y(0) \mid X = x]$$

Here, $Y(1)$ and $Y(0)$ are the outcomes a person would experience *with* and *without* treatment, and $X$ represents their individual characteristics (covariates). Since we can never observe both outcomes for the same person, we need clever estimation strategies.

What makes meta-learners appealing is their flexibility: rather than building specialized causal models from scratch, they *wrap around* any standard machine learning model — random forests, gradient boosting, neural networks — and repurpose it for causal estimation.

> **Key assumption:** All meta-learners require *unconfoundedness* — that once we condition on observed covariates $X$, treatment assignment is effectively random:

$$(Y(1), Y(0)) \perp T \mid X$$

> This is the causal foundation; meta-learners then estimate *how* effects vary across individuals.

## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.

In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython

## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Check and Install Required R Packages

Following R packages are required to run the meta-learner notebooks in this section. If any of these packages are not installed, you can install them using the code below:

`tidyverse`, `grf`, `RCausalML`, `ranger`, `xgboost`, `ggplot2`, `patchwork`, `knitr`, `kableExtra`

In [ ]:
%%R
packages <- c(
  "tidyverse",
  "grf",
  "RCausalML",
  "ranger",
  "xgboost",
  "ggplot2",
  "patchwork",
  "knitr",
  "kableExtra"
)

In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib='drive/My Drive/R/')[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='drive/My Drive/R/')

### Verify Installation

In [ ]:
%%R
# set library path
.libPaths('drive/My Drive/R')
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))

### Load R Packages

In [ ]:
%%R
invisible(lapply(packages, function(pkg) {
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}))

### Check Loaded Packages

In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])

## The Five Main Meta-Learners

Meta-learners differ in *how* they use the treatment indicator to estimate heterogeneous effects. The five main variants represent a spectrum from maximum simplicity to maximum robustness.

| Meta-Learner | Core Idea | Complexity | Best For |
|---|---|---|---|
| **S-Learner** | Single model with $T$ as a feature | Low | Balanced RCT, small effects |
| **T-Learner** | Separate model per treatment arm | Low | Clean, balanced data |
| **X-Learner** | Cross-fitted imputed effects | Medium | Imbalanced groups |
| **R-Learner** | Residual-on-residual regression | High | Observational data |
| **DR-Learner** | Doubly robust pseudo-outcomes | High | Formal inference needs |

### S-Learner (Single Model)

**Idea:** Train one model on all the data, including the treatment indicator $T$ as a feature.

$$\hat{\tau}_S(x) = \hat{\mu}(x, 1) - \hat{\mu}(x, 0)$$

The CATE is estimated by predicting outcomes under treatment ($T=1$) and control ($T=0$) for each individual, then taking the difference.

**Pros:** Simple to implement; naturally captures interactions between treatment and covariates.

**Cons:** Tree-based models may effectively ignore $T$ if it has weak predictive power on its own, leading to underestimated treatment effects.

### T-Learner (Two Separate Models)

**Idea:** Fit one model for the treated group and another for the control group, then subtract predictions.

$$\hat{\tau}_T(x) = \hat{\mu}_1(x) - \hat{\mu}_0(x)$$

**Pros:** Intuitive; each model focuses entirely on its own group.

**Cons:** Struggles when treatment groups are imbalanced (e.g., 95% control, 5% treated) — the minority-group model is estimated from little data and has high variance.

### X-Learner (Cross-Fitted)

**Idea:** An extension of the T-Learner that borrows information across groups to handle imbalance.

**Steps:**

1. Fit T-Learner models $\hat{\mu}_0$, $\hat{\mu}_1$ as before.

2. Compute *imputed treatment effects* for each individual using the opposite group's model:
   - For treated units: $D_i^1 = Y_i - \hat{\mu}_0(X_i)$
   - For control units: $D_i^0 = \hat{\mu}_1(X_i) - Y_i$

3. Fit models $\hat{\tau}_1$, $\hat{\tau}_0$ on these imputed effects.

4. Combine using the estimated propensity score $g(x) = P(T=1 \mid X=x)$:

$$\hat{\tau}_X(x) = g(x)\,\hat{\tau}_0(x) + (1 - g(x))\,\hat{\tau}_1(x)$$

**Pros:** Highly efficient with imbalanced treatment groups; often outperforms the T-Learner empirically.

**Cons:** Requires an accurate propensity score model and involves multiple fitting stages.

### R-Learner (Residual-on-Residual)

**Idea:** Based on Robinson's (1988) transformation — partial out the effect of $X$ on both the outcome and treatment, then regress the residuals against each other.

**Steps:**

1. Estimate nuisance functions:
   - $\hat{m}(x) = \mathbb{E}[Y \mid X=x]$ (expected outcome)
   - $\hat{e}(x) = P(T=1 \mid X=x)$ (propensity score)

2. Compute residuals: $\tilde{Y}_i = Y_i - \hat{m}(X_i)$ and $\tilde{T}_i = T_i - \hat{e}(X_i)$

3. Estimate $\tau(x)$ by solving:

$$\min_{\tau} \mathbb{E}\!\left[\left(\tilde{Y} - \tau(X)\,\tilde{T}\right)^2\right]$$

**Pros:** *Orthogonal* estimation — errors in the nuisance models don't directly bias $\hat{\tau}$. Achieves strong theoretical guarantees (Nie & Wager, 2021).

**Cons:** More complex to implement; requires reasonably good nuisance estimates.

### DR-Learner (Doubly Robust)

**Idea:** Combine outcome modeling and propensity scores so that the estimator is consistent if *either* component is correctly specified — not necessarily both.

**Steps:**

1. Estimate $\hat{\mu}_0(x)$, $\hat{\mu}_1(x)$, and $\hat{e}(x)$ on one data split.

2. Compute augmented inverse propensity weighting (AIPW) pseudo-outcomes on held-out data:

$$\hat{\phi}_i = \underbrace{\hat{\mu}_1(X_i) - \hat{\mu}_0(X_i)}_{\text{outcome model}} + \underbrace{\frac{T_i(Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)} - \frac{(1-T_i)(Y_i - \hat{\mu}_0(X_i))}{1 - \hat{e}(X_i)}}_{\text{bias correction}}$$

3. Regress $\hat{\phi}_i$ on $X_i$ to get $\hat{\tau}_{DR}(x)$.

**Pros:** Doubly robust — remains consistent even with one misspecified component. Comes with formal inference guarantees (asymptotic normality).

**Cons:** Sensitive to extreme propensity scores (near 0 or 1); requires careful cross-fitting.

## Choosing the Right Meta-Learner

| Scenario | Recommended Approach |
|---|---|
| Balanced RCT | S-Learner (simplicity) or X-Learner (efficiency) |
| Imbalanced treatment groups | X-Learner or R-Learner |
| Observational data with confounding | R-Learner or DR-Learner |
| Need for formal statistical inference | DR-Learner |
| Small or subtle treatment effects | S-Learner |

## Extensions: Beyond Binary Treatment

The CATE framework generalizes naturally to other treatment types:

| Treatment Type | Example | Target Estimand |
|---|---|---|
| **Binary** $T \in \{0,1\}$ | Drug vs. placebo | $\tau(x) = \mathbb{E}[Y(1)-Y(0)\mid X=x]$ |
| **Multi-valued** $T \in \{0,\ldots,K\}$ | Dosage levels | $\tau_k(x) = \mathbb{E}[Y(k)-Y(0)\mid X=x]$ |
| **Continuous** $T \in \mathbb{R}$ | Drug dosage (mg) | $\tau(t,x) = \frac{\partial}{\partial t}\mathbb{E}[Y(t)\mid X=x]$ |

## Scientific Terminology for Beginners

This glossary introduces key scientific terms used in this section with simple examples.

### 1) Meta-Learner

- **Explanation**: A two-stage strategy that wraps any supervised ML model to estimate causal effects.
- **Example**: Using a random forest to predict outcomes under treatment and control, then subtracting.

### 2) CATE (Conditional Average Treatment Effect)

- **Explanation**: How much a treatment affects individuals with specific characteristics $X = x$.
- **Example**: A drug reduces blood pressure by 10 mmHg for patients aged over 60, but only 3 mmHg for younger patients.

### 3) Unconfoundedness

- **Explanation**: All variables that simultaneously affect treatment assignment and the outcome are observed and included in $X$.
- **Example**: If healthier patients are more likely to receive treatment, including a pre-treatment health score in $X$ satisfies unconfoundedness.

### 4) Propensity Score

- **Explanation**: The probability that a unit receives treatment given its covariates: $e(x) = P(T=1 \mid X=x)$.
- **Example**: A patient with several risk factors has a propensity score of 0.8, meaning they have an 80% chance of being prescribed treatment.

### 5) Doubly Robust Estimation

- **Explanation**: An estimator that remains consistent if *either* the outcome model or the propensity score model is correctly specified.
- **Example**: Even if the propensity score is slightly misspecified, a correctly specified outcome model keeps the DR-Learner on track.

### 6) Nuisance Functions

- **Explanation**: Intermediate models (outcome regression, propensity score) fit to remove confounding, not directly of interest themselves.
- **Example**: Estimating $E[Y \mid X]$ is a nuisance step; the CATE $\tau(x)$ is the target.

### 7) Cross-Fitting

- **Explanation**: Fitting nuisance models on one data fold and computing residuals on another, preventing overfitting bias.
- **Example**: Split data into 5 folds; fit the propensity score on folds 2–5, use it to compute residuals on fold 1.

## Comparison of Meta-Learner Properties

| Feature | S-Learner | T-Learner | X-Learner | R-Learner | DR-Learner |
|---|---|---|---|---|---|
| **Number of models** | 1 | 2 | 4 | 3 | 3 + 1 |
| **Uses propensity score** | No | No | Yes | Yes | Yes |
| **Handles imbalance well** | Poor | Poor | Good | Good | Good |
| **Doubly robust** | No | No | No | No | Yes |
| **Formal inference** | No | No | No | Partial | Yes |
| **Implementation complexity** | Low | Low | Medium | High | High |
| **Cross-fitting required** | No | No | No | Recommended | Required |

## Summary

Meta-learners offer a practical, modular approach to estimating heterogeneous treatment effects. Their core appeal is that they repurpose familiar ML tools for causal questions — no specialized architecture required. The tradeoffs are straightforward:

- **Simpler learners (S, T)** work well in clean experimental settings where data are balanced and confounding is minimal.
- **X-Learner** is the go-to when treatment groups are imbalanced — it borrows strength across arms via imputed counterfactuals.
- **R-Learner** provides orthogonal estimation that is robust to nuisance model errors, making it ideal for observational data with measured confounders.
- **DR-Learner** adds doubly robust protection and asymptotic normality, enabling formal hypothesis testing and confidence intervals.

In practice, no single meta-learner dominates in all settings. The choice depends on the degree of treatment imbalance, the quality of available confounders, computational budget, and whether formal inference is needed. The subsequent notebooks in this section demonstrate each meta-learner in action on real and simulated datasets.

## Resources

1. **Künzel, S. R., Sekhon, J. S., Bickel, P. J., & Yu, B. (2019).** *Metalearners for estimating heterogeneous treatment effects using machine learning.* PNAS. (Original meta-learner framework introducing S-, T-, and X-Learners.)

2. **Nie, X., & Wager, S. (2021).** *Quasi-oracle estimation of heterogeneous treatment effects.* Biometrika. (Theoretical foundation for the R-Learner.)

3. **Chernozhukov, V., et al. (2018).** *Double/Debiased Machine Learning for Treatment and Structural Parameters.* The Econometrics Journal. (Foundation for DR-Learner and cross-fitting.)

4. **Kennedy, E. H. (2023).** *Semiparametric doubly robust targeted double machine learning: a review.* Statistical Science. (Comprehensive review of doubly robust methods.)

5. **Athey, S., & Imbens, G. W. (2016).** *Recursive partitioning for heterogeneous causal effects.* PNAS. (Tree-based foundations for meta-learners.)

6. **RCausalML R Package** — Implementation of meta-learners used in this tutorial series.

7. **grf R Package** — Generalized Random Forests, including R-Learner variants. https://grf-labs.github.io/grf/